# 实时扩声（话筒 → 系统 → 音响）

**只有代码格 3 会发声。** 格 1、2、4 都不发声。

现场要求：麦克风与音响按标定位置摆好，麦克风接 `LINE_IN`、VCC 3.3 V、GND 接好；
音响接 `HP+MIC`，能立即断电；测试期间不要移动夹具、不要碰桌面。

格 3 运行前请把世界安静下来，运行开始后**再从格 3 之后的窗口气说话**——
因为音响在窗口开始的瞬间就会开始扩声。

## 代码格 1（不发声）：核对文件版本并导入
预期输出最后一行是 `LIVE_READY`。出现任何 `MISMATCH` 就停止，不要执行格 2。

In [ ]:
import hashlib, json
from pathlib import Path
EXPECT = {
    'audio_replay.bit': '8b2426f4fbcf27425731899557b179d2aea8a4ec2fa26dcc11b60102ecb45339',
    'audio_replay.hwh': '3873b16b0e273945be8acff0cdebcfbc6e41a6b694d179d3b62955fd452505bb',
    'model_room_512.json': '07fae6af19b1825044b194672d2f9d1ab4d1897981817d48e36916a7e30bfddb',
    'live_pa.py': 'a531ffe52ae8c87173fa71037a212324003684789794826cb99bd978934e7458',
}
for name, want in EXPECT.items():
    got = hashlib.sha256(Path(name).read_bytes()).hexdigest()
    print(('OK   ' if got == want else 'MISMATCH ') + name)
    assert got == want, name
from live_pa import live_pa
print('LIVE_READY')

## 代码格 2（不发声）：设定本轮参数
默认：**旁路**、增益 **−18 dB**、每段 8 秒、共 2 段。只有这一段会写入磁盘。
把 `BYPASS` 改成 `False` 就是带抵消的扩声；`GAIN_DB` 越小越安全。

In [ ]:
from pathlib import Path
BYPASS = True          # True = 旁路扩声；False = 带抵消扩声
GAIN_DB = -18.0        # 千万别先往上调；改大必须有人值守并准备断电
SECONDS = 8.0          # 每段秒数，上限 9.5
BURSTS = 2             # 段数
OUTDIR = Path('/home/xilinx/live_run')
OUTDIR.mkdir(exist_ok=True)
print('CONFIG', dict(bypass=BYPASS, gain_db=GAIN_DB, seconds=SECONDS, bursts=BURSTS))
print('STARTS_IN_A_FEW_SECONDS_AFTER_CELL3_RUNS')

## 代码格 3（发声）：实时扩声
**执行后立刻开始说话**，正常音量、离麦克风约 0.5 m。
预期：音响里听到自己的声音；约 16 秒后自动静音并恢复官方位流。
出现啸叫或不舒适：直接切断音响电源，或点中断（■）停止本格。

预期输出：每段一行 JSON，最后 `STOCK_BASE_RESTORED_AND_CODEC_MUTED`。

In [ ]:
summary = live_pa(
    fixture_id='room_v1_20261021',
    model_path='model_room_512.json',
    bypass=BYPASS,
    gain_db=GAIN_DB,
    seconds=SECONDS,
    bursts=BURSTS,
    hp_volume=40,
    speaker_confirmed=True,
    local_capture_confirmed=True,
    output_dir=OUTDIR,
    label='live',
)
print('DONE', summary['bursts'], 'windows')

## 代码格 4（不发声）：汇总与留证
预期：打印每段的麦克风电平与保护计数，并给出结果包路径。

In [ ]:
rows = []
for w in summary['windows']:
    rows.append({k: w.get(k) for k in ('burst', 'frames', 'mic_rms_dbfs', 'mic_peak_dbfs',
                                       'status', 'guard_tripped', 'prediction_saturations',
                                       'residual_saturations', 'gain_saturations', 'capture')})
import json as _json
for row in rows:
    print(_json.dumps(row, ensure_ascii=False))
print('RESULT_PACKAGE', summary['fixture_id'], summary['bypass'], summary['gain_db'])
print('FILES_IN', OUTDIR)